# GTEx Tissue Classifier, Real Data Tutorial
This tutorial will outline the complete process for using the command line tool on real data, from setup and installation, to commands and figure outputs. Following this tutorial, you will be able to take your own RNA sequencing data, prepare it for tool input, and recieve tissue origin predictions for your samples. The biological question we are attempting to answer is if we can identify the tissue origin of an RNA sequencing sample based on its gene expression, when compared to the GTEx tissue database.  

## Setup and Installation
To avoid dependency conflicts, a pre-configured Conda environment is included with this tool, located at `environment.yml`. Please review the dependencies you will be installing ahead of time for your safety. In this step we assume you are accessing this notebook by succesfully installing the tool. Verify you have activated the gtex-env Conda environment and installed the package in editable mode. 

## Data Preparation
Often, the data found on GEO is not already formatted to our required specifications. This can be handled by inspecting the provided file and writing a few lines of code to extract the gene names and transcripts per million (TPM) values for a given sample. In practice, you would want to do a TPM normalization during your processing steps in preparation for using this tool. First, download the [GSE126848_Gene_counts_raw.txt.gz](https://ftp.ncbi.nlm.nih.gov/geo/series/GSE126nnn/GSE126848/suppl/GSE126848%5FGene%5Fcounts%5Fraw%2Etxt%2Egz) file and extract it to the `tutorials` folder. For the GSE126848 liver RNA seq data, the counts per million (CPM) values are used in place of TPM values, because the tool should still identify tissues without gene length normalization. It will likely be more accurate, however, if TPM values are instead supplied. The below code should provide a starting point for processing similar files:

In [ ]:
def format_geo_for_testing(input_filepath: str, output_filepath: str) -> None:
    """
    Parses a raw GEO counts matrix, isolates a sample, calculates CPM,
    and formats it to appropriate input format.
    """
    print(f"Loading raw matrix: {input_filepath}")
    df = pd.read_csv(input_filepath, sep='\t')
    
    # Isolate the first column containing Ensembl IDs and the index of the desired patient sample, in this case index 1 is a healthy control patient
    gene_col = df.columns[0]
    target_sample = df.columns[1]
    test_df = df[[gene_col, target_sample]].copy()
    
    # Calculate CPM
    total_reads = test_df[target_sample].sum()
    test_df['CPM'] = (test_df[target_sample] / total_reads) * 1e6
    
    # Keep only the CPM values and rename to TPM to pass input validation
    final_df = test_df[[gene_col, 'CPM']].copy()
    final_df.columns = ['Gene', 'TPM']
    
    # Save the file
    final_df.to_csv(output_filepath, index=False)
    print(f"Test sample saved to {output_filepath}")
format_geo_for_testing('GSE126848_Gene_counts_raw.txt', 'mock_liver_sample.csv')

## Running the GTEx Tissue Classifier
Now that the sample data is prepared and properly formatted, it is ready to be passed into the command line input handled by `architecture.py`. \
The following arguments are used: \
'--mode real' - This tells the tool we want to run it with real data, instead of it providing sample data for us. \
'--input' - Path to your input sample, in our case 'mock_liver_sample.csv' \
'--output' - Path where you want the output figures. \
'--num-genes' - Select the number of most variable genes you want to isolate from the GTEx database to compare your sample against. This value is set at 2000 by default to be comprehensive, although some tissues can be identified with less highly variable genes. However, this is not recommended because the accuracy drops with fewer genes.

In [ ]:
# Run the classifier
!python -m gtex_tissue_classifier.architecture \
    --mode real \
    --input 'mock_liver_sample.csv' \
    --output 'GSE126848_sample_0869' \
    --num-genes 2000

## Visualizing the results
The first output file is the summary report that shows the most likely tissue that the sample came from, and below it is the calculated spearman correlation. Additionally, a summary table of the top 5 closest matches is outputted, with the tissue name and correlation values provided. The image below is a heatmap showing the top genes that are uniquely highly expressed in liver tissue. The heatmap compares side-by-side the expression of these genes in the sample and in the GTEx reference data. 

In [ ]:
import os
from IPython.display import display, HTML, Image

# Load files from our output folder
output_path = "GSE126848_sample_0869"

# Load the HTML summary
report_path = os.path.join(output_path, "summary_report.html")
if os.path.exists(report_path):
    with open(report_path, "r") as f:
        display(HTML(f.read()))
else:
    print(f"Error: Could not find {report_path}")

# Load the heatmap
heatmap_path = os.path.join(output_path, "tissue_rankings.png")
if os.path.exists(heatmap_path):
    display(Image(filename=heatmap_path))
else:
    print(f"Error: Could not find {heatmap_path}")

## Interpreting the results
Given that our sample has a robust correlation of 0.8874 to the predicted tissue of origin, liver, we have a high degree of confidence that the classifier was able to accurately predict our sample tissue. Futhermore, the next three closest matches were all other types of liver cells, which adds to our confidence in predicting liver tissue. Another key metric is the delta between the correlation of the predicted tissue and the next best guess outside of liver, which in this case was kidney cortex. The difference between the two correlations is ~0.37, meaning the tool received a robust signal that the input sample came from liver tissue. From the heatmap, we can see the most highly expressed gene in the reference tissue is also the most highly expressed in the input sample, which is another indicator of confidence in the match. 